# Laboratorio 6 — Análisis de redes sociales (YouTube)
### CC3084 — Data Science | Universidad del Valle de Guatemala | Semestre II — 2026

**Avance:** Incisos 1 y 2 (Carga/integración de datos, y Calidad/limpieza/preprocesamiento)

Este notebook cubre:
1. Carga, comprensión e integración de los datos (`youtube_videos.csv`, `youtube_comments.csv`)
2. Calidad, limpieza y preprocesamiento (diagnóstico, normalización de IDs, conversión de conteos, limpieza de texto)

> Nota metodológica: los datos **no permiten identificar quién respondió a quién**. `reply_count` indica cuántas respuestas recibió un comentario principal, pero no identifica a los autores de esas respuestas ni su contenido, por lo que en ningún punto de este análisis se interpreta como una arista entre usuarios.


In [1]:
# Librerías generales
import pandas as pd
import numpy as np
import ast
import re
import json
import unicodedata

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)

RANDOM_STATE = 42


## 1. Carga, comprensión e integración de los datos

### 1.1 Carga de los archivos `youtube_videos.csv` y `youtube_comments.csv`

In [2]:
videos = pd.read_csv('data/youtube_videos.csv')
comments = pd.read_csv('data/youtube_comments.csv')

print(f"videos: {videos.shape[0]} filas x {videos.shape[1]} columnas")
print(f"comments: {comments.shape[0]} filas x {comments.shape[1]} columnas")


videos: 293 filas x 20 columnas
comments: 406 filas x 17 columnas


In [3]:
videos.head(3)

,video_id,title,channel_name,channel_id,source_query,source_group,dataset_sources,channel_handle,published_time,view_count_text,description_snippet,video_url,query_hits,keywords,description,view_count,publish_date,upload_date,category,owner_handle
0,-5puKGEqcUc,INSIVUMEH pronostica incremento de lluvias para el fin de semana en Guatemala,T13 Noticias Guatemala,UCq0Cm-3SKthEySQc2JZBi1A,guatemala lluvias,topic,youtube_guatemala.csv | youtube_guatemala_lab.csv | youtube_guatemala_plus.csv,/@T13NoticiasGuatemala,hace 2 días,"2,390 vistas",El Departamento de Pronóstico de INSIVUMEH prevé un incremento en las lluvia...,https://www.youtube.com/watch?v=-5puKGEqcUc,"[""guatemala lluvias""]","[""Canícula prolongada"", ""Chapin tv"", ""Fenómeno del Niño"", ""Guatemala"", ""Lluv...",El Departamento de Pronóstico de INSIVUMEH prevé un incremento en las lluvia...,2357,2026-08-28T22:00:20-07:00,2026-08-28T22:00:20-07:00,News & Politics,/@T13NoticiasGuatemala
1,-E7OPOLjMug,BERNARDO ARÉVALO CALIFICA CAMBIO EN EL MP COMO EL FIN DE UNA ETAPA DE DETERI...,IDocumenta,UCgItjn_ZWFWcv1MlpKIkqgQ,@GobiernodelaRepublicadeGuatema,topic,youtube_target_channels.csv,/@iDocumenta,hace 3 meses,4 vistas,"El presidente de Guatemala, Bernardo Arévalo, calificó el cambio de fiscal g...",https://www.youtube.com/watch?v=-E7OPOLjMug,"[""@GobiernodelaRepublicadeGuatema""]",[],"El presidente de Guatemala, Bernardo Arévalo, calificó el cambio de fiscal g...",4,2026-05-13T16:00:03-07:00,2026-05-13T16:00:03-07:00,People & Blogs,/@iDocumenta
2,-KDglrIzRKo,¡HISTÓRICO! Mexico recupera petróleo robado por Guatemala... 🔔,México Poder,UC-DpoeBbCMOMOH1-j71KPqA,guatemala noticias,topic,youtube_guatemala.csv | youtube_guatemala_lab.csv | youtube_guatemala_plus.csv,/@M%C3%A9xicoPoder,hace 1 día,"29,736 vistas",México #PetróleoMexicano #SoberaníaEnergética #Pemex #NoticiasMéxico #Fronte...,https://www.youtube.com/watch?v=-KDglrIzRKo,"[""guatemala noticias""]","[""Mexico recupera petroleo"", ""petroleo robado Guatemala"", ""huachicol fronter...",#México #PetróleoMexicano #SoberaníaEnergética #Pemex #NoticiasMéxico #Front...,29736,2026-08-29T17:00:38-07:00,2026-08-29T17:00:38-07:00,People & Blogs,/@M%C3%A9xicoPoder


In [4]:
comments.head(3)

,video_id,comment_id,video_title,channel_name,channel_id,author_name,author_channel_id,text,source_query,source_group,dataset_sources,author_handle,published_text,like_count_text,reply_count,is_pinned,viewer_rating
0,j43HgwYFKfk,Ugw-J65a1iYL9hqhELh4AaABAg,La cooptación de Walter Mazariegos en la USAC,Quorum,UCE4rsXcgDb6e1-a9iTbWzfg,@MarcosCarillo-b1r,UCdFlugHJJa4l3YqWuNRmvXw,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,@quorumgt,topic,j43HgwYFKfk_out_comments.csv | quorum_complete_comments.csv | youtube_guatem...,/@MarcosCarillo-b1r,hace 6 meses,,0,False,NaN
1,06mFNPU0aB8,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,Capturan a presuntos delincuentes disfrazados de mujer señalados de cometer ...,Noti7,UCVpSRoZgngfSL03Nlbjtq9A,@RaulPerez-cw2vi,UCvl1tzQeBeGy6efPTRJXSCw,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic...",guatemala noticias,topic,youtube_guatemala_comments.csv | youtube_guatemala_plus_comments.csv,/@RaulPerez-cw2vi,hace 2 semanas,,0,False,NaN
2,j43HgwYFKfk,Ugw0xaOb2CYXXoudtwJ4AaABAg,La cooptación de Walter Mazariegos en la USAC,Quorum,UCE4rsXcgDb6e1-a9iTbWzfg,@iamjimalesssa,UCRAquv8el-tQ30bN7MlmySQ,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e...",@quorumgt,topic,j43HgwYFKfk_out_comments.csv | quorum_complete_comments.csv | youtube_guatem...,/@iamjimalesssa,hace 1 año,4,0,False,NaN


### 1.2 Unidad de observación, llave primaria y variables relevantes

**`youtube_videos.csv`**
- **Unidad de observación:** un video de YouTube recuperado durante la recolección.
- **Llave primaria:** `video_id` (identificador único de video en YouTube; verificado sin duplicados abajo).
- **Variables relevantes:**
  - Identificación del video/canal: `video_id`, `channel_id` (preferido sobre `channel_name`, que puede repetirse), `channel_handle`.
  - Contenido: `title`, `description`, `description_snippet`, `keywords`.
  - Popularidad: `view_count` (numérica, apta para cálculo), `view_count_text` (texto crudo de YouTube).
  - Procedencia/muestreo: `source_query`, `source_group`, `query_hits`, `dataset_sources`.
  - Temporalidad: `publish_date` / `upload_date` (ISO 8601, idénticas entre sí) y `published_time` (texto relativo, dependiente del momento de scraping).
  - Categoría: `category` (categoría asignada por YouTube).

**`youtube_comments.csv`**
- **Unidad de observación:** un comentario **principal** publicado en un video (no incluye respuestas/replies).
- **Llave primaria:** `comment_id` (identificador único de comentario; verificado sin duplicados abajo).
- **Llave foránea:** `video_id`, que conecta cada comentario con su video en `youtube_videos.csv`.
- **Variables relevantes:**
  - Identificación del autor: `author_channel_id` (identificador único y estable del autor; debe preferirse a `author_name`/`author_handle`, que pueden cambiar o repetirse).
  - Contenido: `text` (variable principal para tópicos y sentimiento).
  - Popularidad/interacción: `like_count_text` (texto, requiere limpieza), `reply_count` (numérica, **no** identifica autores de las respuestas).
  - Procedencia: `source_query`, `source_group`, `dataset_sources`.
  - Temporalidad: `published_text` (texto relativo).
  - Variables no utilizables: `is_pinned` (constante, siempre `False`) y `viewer_rating` (100% nula).


In [5]:
# Verificación de llaves primarias (sin duplicados)
print("Duplicados en video_id (videos):   ", videos['video_id'].duplicated().sum())
print("Duplicados en comment_id (comments):", comments['comment_id'].duplicated().sum())


Duplicados en video_id (videos):    0
Duplicados en comment_id (comments): 0


### 1.3 Relación entre canal, video, autor del comentario, comentario, categoría y consulta de búsqueda

La estructura relacional del conjunto de datos es la siguiente:

- Un **canal** (`channel_id`) puede publicar **muchos videos** (relación 1 → N). Cada video pertenece a un único canal.
- Un **video** (`video_id`) puede recibir **muchos comentarios** (relación 1 → N, vía `video_id` como llave foránea en `comments`). En este conjunto, 274 de los 293 videos no tienen comentarios asociados (cobertura parcial — ver limitaciones).
- Un **autor de comentario** (`author_channel_id`) puede publicar **comentarios en varios videos distintos** (relación N → M entre autores y videos), lo cual es precisamente la base de la red bipartita autor-video que se construirá en incisos posteriores.
- Cada video tiene asignada una **categoría** de YouTube (`category`), una variable categórica que describe el tipo de contenido (p. ej. *News & Politics*, *People & Blogs*).
- La **consulta de búsqueda** (`source_query`) describe el *procedimiento de muestreo*: el término o canal utilizado para encontrar el video durante la recolección. Un mismo video puede coincidir con varias consultas (`query_hits`), y `source_group` indica si el video se encontró por tema (`topic`), por ser un canal de gobierno (`official_gov`) o por canal específico (`channel`). **Importante:** la consulta de búsqueda describe *cómo* se encontró el contenido, no necesariamente *de qué trata* — dos videos con la misma consulta pueden ser de temas distintos, y el mismo canal se encuentra tanto en `videos` como en `comments` a través de `channel_id`, que es consistente entre ambos archivos (validado abajo).

En síntesis: **canal → video → comentario → autor**, con `category` y `source_query` como atributos descriptivos de video, no como entidades con llave propia.


In [6]:
# Validación: el channel_id asociado a un comentario (vía su video) es
# consistente con el channel_id del video en 'videos'
chk = comments.merge(videos[['video_id', 'channel_id']], on='video_id',
                      suffixes=('_comment_video', '_videos_file'))
inconsistencias = (chk['channel_id_comment_video'] != chk['channel_id_videos_file']).sum()
print(f"Inconsistencias channel_id (comments vs videos) tras el join: {inconsistencias} / {len(chk)}")

print(f"\nCanales únicos en videos.csv: {videos['channel_id'].nunique()}")
print(f"Autores únicos (author_channel_id) en comments.csv: {comments['author_channel_id'].nunique()}")


Inconsistencias channel_id (comments vs videos) tras el join: 0 / 406

Canales únicos en videos.csv: 97
Autores únicos (author_channel_id) en comments.csv: 332


### 1.4 Integración de los conjuntos mediante `video_id`

In [7]:
# Integración: cada comentario se enriquece con los atributos de su video
df = comments.merge(
    videos,
    on='video_id',
    how='left',
    suffixes=('_comment', '_video')
)

comentarios_asociados = df['title'].notna().sum()  # 'title' solo existe si hubo match con videos
print(f"Comentarios totales:              {len(comments)}")
print(f"Comentarios asociados a un video: {comentarios_asociados}")
print(f"Comentarios sin video asociado:   {len(comments) - comentarios_asociados}")


Comentarios totales:              406
Comentarios asociados a un video: 406
Comentarios sin video asociado:   0


**Resultado:** los 406 comentarios (100%) pudieron asociarse correctamente a un video mediante `video_id`, lo cual es consistente con la validación de la sección 1.3 (todo `video_id` presente en `comments.csv` existe en `videos.csv`). El DataFrame integrado `df` tiene una fila por comentario, enriquecida con los atributos del video correspondiente (título, canal, categoría, visualizaciones, etc.), y se usará como base para el análisis exploratorio del inciso 3.

In [8]:
print(f"df integrado: {df.shape[0]} filas x {df.shape[1]} columnas")
df[['comment_id', 'video_id', 'author_channel_id', 'title', 'category', 'view_count']].head(5)

df integrado: 406 filas x 36 columnas


,comment_id,video_id,author_channel_id,title,category,view_count
0,Ugw-J65a1iYL9hqhELh4AaABAg,j43HgwYFKfk,UCdFlugHJJa4l3YqWuNRmvXw,La cooptación de Walter Mazariegos en la USAC,News & Politics,10156
1,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,06mFNPU0aB8,UCvl1tzQeBeGy6efPTRJXSCw,Capturan a presuntos delincuentes disfrazados de mujer señalados de cometer ...,News & Politics,6692
2,Ugw0xaOb2CYXXoudtwJ4AaABAg,j43HgwYFKfk,UCRAquv8el-tQ30bN7MlmySQ,La cooptación de Walter Mazariegos en la USAC,News & Politics,10156
3,Ugw0xgUc2ISpBr5_T654AaABAg,j43HgwYFKfk,UCkbsS_3D-pvg1iHOld9uvIg,La cooptación de Walter Mazariegos en la USAC,News & Politics,10156
4,Ugw1ZzA21njWhaqQQTh4AaABAg,OkXlHx0hx-8,UCOLHH4ZpMxPYF-Q6e6wvn5A,EE.UU. envía a mexicanos deportados a Guatemala antes de su regreso a México...,News & Politics,14200


## 2. Calidad, limpieza y preprocesamiento

### 2.1 Diagnóstico inicial de calidad

In [9]:
def diagnostico_calidad(df_, nombre):
    print(f"{'='*70}\nDIAGNÓSTICO: {nombre}\n{'='*70}")
    print(f"Dimensiones: {df_.shape[0]} filas x {df_.shape[1]} columnas\n")

    print("Tipos de variables:")
    print(df_.dtypes.value_counts(), "\n")

    print("Valores faltantes por columna (solo columnas con > 0):")
    faltantes = df_.isnull().sum()
    faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
    if len(faltantes):
        print(faltantes)
    else:
        print("(ninguna)")
    print()

    print(f"Filas duplicadas (todas las columnas): {df_.duplicated().sum()}")
    return faltantes

falt_videos = diagnostico_calidad(videos, "youtube_videos.csv")


DIAGNÓSTICO: youtube_videos.csv
Dimensiones: 293 filas x 20 columnas

Tipos de variables:
str      19
int64     1
Name: count, dtype: int64 

Valores faltantes por columna (solo columnas con > 0):
description            26
description_snippet    25
view_count_text        13
published_time         13
dtype: int64

Filas duplicadas (todas las columnas): 0


In [10]:
falt_comments = diagnostico_calidad(comments, "youtube_comments.csv")

DIAGNÓSTICO: youtube_comments.csv
Dimensiones: 406 filas x 17 columnas

Tipos de variables:
str        14
int64       1
bool        1
float64     1
Name: count, dtype: int64 

Valores faltantes por columna (solo columnas con > 0):
viewer_rating    406
dtype: int64

Filas duplicadas (todas las columnas): 0


In [11]:
# Variables constantes (una sola categoría en todo el dataset)
def variables_constantes(df_, nombre):
    constantes = [c for c in df_.columns if df_[c].nunique(dropna=False) == 1]
    print(f"Variables constantes en {nombre}: {constantes}")
    return constantes

const_videos = variables_constantes(videos, "videos")
const_comments = variables_constantes(comments, "comments")


Variables constantes en videos: []
Variables constantes en comments: ['is_pinned', 'viewer_rating']


In [12]:
# Consistencia entre identificadores, nombres y handles
print("videos.csv:")
print(f"  channel_name -> ¿biyectivo con channel_id?  ",
      (videos.groupby('channel_name')['channel_id'].nunique() > 1).sum() == 0
      and (videos.groupby('channel_id')['channel_name'].nunique() > 1).sum() == 0)
print(f"  ¿owner_handle == channel_handle en el 100%%?  {(videos['owner_handle'] == videos['channel_handle']).all()}")
print(f"  ¿publish_date == upload_date en el 100%%?     {(videos['publish_date'] == videos['upload_date']).all()}")

print("\ncomments.csv:")
print(f"  author_name -> ¿biyectivo con author_channel_id?  ",
      (comments.groupby('author_name')['author_channel_id'].nunique() > 1).sum() == 0
      and (comments.groupby('author_channel_id')['author_name'].nunique() > 1).sum() == 0)


videos.csv:
  channel_name -> ¿biyectivo con channel_id?   True
  ¿owner_handle == channel_handle en el 100%%?  True
  ¿publish_date == upload_date en el 100%%?     True

comments.csv:
  author_name -> ¿biyectivo con author_channel_id?   True


In [13]:
# Valores atípicos: view_count (videos) y reply_count (comments)
print("view_count (videos) - percentiles:")
print(videos['view_count'].quantile([0, .01, .25, .5, .75, .95, .99, 1]).astype(int))

q1, q3 = videos['view_count'].quantile([.25, .75])
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
atipicos_view = (videos['view_count'] > lim_sup).sum()
print(f"\nLímite superior IQR: {lim_sup:,.0f}  ->  {atipicos_view} videos por encima ({atipicos_view/len(videos):.1%})")

print("\nreply_count (comments) - distribución:")
print(comments['reply_count'].value_counts().sort_index())


view_count (videos) - percentiles:
0.00          2
0.01          3
0.25        215
0.50       1175
0.75       7465
0.95     130466
0.99     523972
1.00    8190449
Name: view_count, dtype: int64

Límite superior IQR: 18,340  ->  49 videos por encima (16.7%)

reply_count (comments) - distribución:
reply_count
0    376
1     21
2      3
3      4
5      1
7      1
Name: count, dtype: int64


**Hallazgos del diagnóstico:**
- No hay filas duplicadas completas ni `video_id`/`comment_id` duplicados en ninguno de los dos archivos.
- Los valores faltantes se concentran en variables de texto secundarias de `videos.csv` (`published_time`, `view_count_text`: 13 nulos cada una; `description_snippet`: 25; `description`: 26) y en `viewer_rating` de `comments.csv` (406/406, es decir, 100%).
- `is_pinned` (comments) es **constante** (`False` en todos los registros): no aporta variabilidad y se descarta del análisis.
- `channel_name` ↔ `channel_id` y `author_name` ↔ `author_channel_id` son consistentes (relación 1 a 1) en este conjunto, aunque por definición del laboratorio se prefieren los identificadores (`channel_id`, `author_channel_id`) por ser más estables.
- `owner_handle`/`channel_handle` y `publish_date`/`upload_date` son idénticas al 100%, por lo que son redundantes.
- `view_count` presenta una distribución muy asimétrica (long tail): la mediana es 1,175 vistas pero el máximo es 8,190,449; usando el criterio de rango intercuartílico hay videos con visualizaciones marcadamente atípicas (viral outliers), consistentes con la naturaleza de YouTube y no con errores de captura.
- `reply_count` está dominado por ceros (mediana 0), con un máximo de 7 respuestas por comentario.


### 2.2 Variables problemáticas o de uso delicado

| Variable | Problema | Tratamiento |
|---|---|---|
| `is_pinned` (comments) | Constante (100% `False`) | Se documenta y se excluye del análisis; no discrimina nada. |
| `viewer_rating` (comments) | 100% nula | Se descarta por completo, no puede usarse. |
| `channel_name`, `author_name`, `channel_handle`, `author_handle` | Pueden cambiar o repetirse en el tiempo; no son identificadores únicos | Se conservan **solo** como etiquetas visuales/de despliegue; para construir nodos de red y agrupar se usan siempre `channel_id` / `author_channel_id`. |
| `published_time` (videos), `published_text` (comments) | Fechas **relativas** ("hace 2 días"), dependientes del momento de recolección — no son fechas exactas reconstruibles con precisión | Se documentan como no aptas para análisis temporal fino; para videos existe la alternativa exacta `publish_date` (ISO 8601), que se usa en su lugar. Para comentarios no existe una fecha exacta equivalente, por lo que cualquier análisis temporal de comentarios queda limitado a la categoría relativa observada. |
| `upload_date` (videos) | 100% idéntica a `publish_date` | Redundante; se conserva `publish_date` como referencia y se documenta la redundancia (no se elimina la columna original del CSV, solo no se usa duplicada). |
| `owner_handle` (videos) | 100% idéntica a `channel_handle` | Redundante, mismo tratamiento que el punto anterior. |
| `view_count_text`, `like_count_text` | Texto con separadores de miles y sufijos (`"2,390 vistas"`), o vacíos (`" "`) | Se convierten a variables numéricas (`view_count` ya viene numérica en videos; `like_count_text` se limpia y convierte en la sección 2.4). |
| `query_hits`, `keywords`, `dataset_sources` (videos) y `dataset_sources` (comments) | Texto con estructura de lista (`'["a", "b"]'` o separado por `|`) | Se parsean a listas de Python (`ast.literal_eval` / `split('|')`) antes de cualquier análisis de frecuencia. |
| `text` (comments) | Contiene URLs, menciones, hashtags, emojis y errores ortográficos típicos de comentarios de YouTube | Se conserva el texto original (`texto_original`) para auditoría/sentimiento, y se genera una versión limpia (`texto_limpio`) para análisis de tópicos/frecuencia (sección 2.5–2.6). |
| `view_count` (videos) | Fuertemente asimétrica, con outliers de alta visibilidad ("virales") | No se eliminan los outliers: son observaciones reales y relevantes para el análisis de popularidad; se documentan y se usan escalas apropiadas (p. ej. logarítmica) al graficar. |
| `reply_count` (comments) | Podría malinterpretarse como relación entre usuarios | Se usa **únicamente** como conteo agregado de respuestas al comentario principal; explícitamente **no** se construye ninguna arista de red a partir de esta variable, siguiendo la advertencia del enunciado. |


### 2.3 Normalización de identificadores y nombres

Se conservan los identificadores (`channel_id`, `video_id`, `comment_id`, `author_channel_id`) como llaves de análisis, y se normalizan los nombres/handles solo con fines de **despliegue** (nunca se sustituyen los IDs por nombres visibles).


In [14]:
def normalizar_texto_visible(s):
    """Normaliza espacios en textos visibles (nombres/handles) sin alterar
    su contenido semántico ni usarlos como identificador."""
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = re.sub(r'\s+', ' ', s)
    return s

for col in ['channel_name', 'channel_handle']:
    videos[col] = videos[col].apply(normalizar_texto_visible)

for col in ['channel_name', 'author_name', 'author_handle']:
    comments[col] = comments[col].apply(normalizar_texto_visible)

# Los identificadores se dejan tal cual (no se recortan ni transforman en contenido),
# solo se verifica que no tengan espacios accidentales
for col in ['video_id', 'channel_id']:
    videos[col] = videos[col].str.strip()
for col in ['video_id', 'comment_id', 'channel_id', 'author_channel_id']:
    comments[col] = comments[col].str.strip()

print("IDs normalizados (sin espacios). Ejemplo de nombres/handles tras normalizar:")
videos[['channel_id', 'channel_name', 'channel_handle']].head(3)


IDs normalizados (sin espacios). Ejemplo de nombres/handles tras normalizar:


,channel_id,channel_name,channel_handle
0,UCq0Cm-3SKthEySQc2JZBi1A,T13 Noticias Guatemala,/@T13NoticiasGuatemala
1,UCgItjn_ZWFWcv1MlpKIkqgQ,IDocumenta,/@iDocumenta
2,UC-DpoeBbCMOMOH1-j71KPqA,México Poder,/@M%C3%A9xicoPoder


### 2.4 Conversión a numérico de variables de conteo almacenadas como texto

Se documentan dos casos:
- **`view_count_text`** (videos): sufijo `" vistas"` y separador de miles `,`. La columna `view_count` (ya numérica) se usa como fuente de verdad; se valida que ambas sean consistentes donde `view_count_text` no es nulo.
- **`like_count_text`** (comments): puede venir vacía (representada como un único espacio `' '`), lo cual corresponde a comentarios sin "me gusta" registrados (0 likes) y no a un dato faltante real — se documenta la decisión de imputar 0.


In [15]:
def parsear_conteo(texto):
    """Convierte un conteo tipo YouTube ('2,390 vistas', ' ', '45') a entero.
    Reglas documentadas:
      - Elimina separadores de miles (',').
      - Elimina sufijos no numéricos (p. ej. ' vistas').
      - Cadenas vacías o solo espacios -> 0 (ausencia de "me gusta" registrados).
      - Cualquier valor no numérico tras la limpieza -> NaN (valor inválido, se reporta).
    """
    if pd.isna(texto):
        return np.nan
    t = str(texto).strip()
    if t == '':
        return 0
    t = re.sub(r'[^\d]', '', t)  # deja solo dígitos
    if t == '':
        return 0
    return int(t)

# --- videos: validar consistencia entre view_count_text y view_count ---
videos['view_count_from_text'] = videos['view_count_text'].apply(parsear_conteo)
mask_comparable = videos['view_count_from_text'].notna()
inconsistentes = (videos.loc[mask_comparable, 'view_count_from_text']
                   != videos.loc[mask_comparable, 'view_count']).sum()
print(f"view_count_text vs view_count: {inconsistentes} inconsistencias de {mask_comparable.sum()} comparables")
print(f"view_count_text nulo (sin equivalente en texto): {videos['view_count_text'].isna().sum()} filas "
      f"(se usa igualmente view_count, que sí está completa)")

# --- comments: like_count ---
comments['like_count'] = comments['like_count_text'].apply(parsear_conteo)
vacios = (comments['like_count_text'].str.strip() == '').sum()
invalidos = comments['like_count'].isna().sum()
print(f"\nlike_count_text vacíos/espacio -> imputados a 0: {vacios}")
print(f"like_count_text no convertibles (valores inválidos): {invalidos}")
print(f"like_count resultante - describe():")
print(comments['like_count'].describe())


view_count_text vs view_count: 53 inconsistencias de 280 comparables
view_count_text nulo (sin equivalente en texto): 13 filas (se usa igualmente view_count, que sí está completa)

like_count_text vacíos/espacio -> imputados a 0: 189
like_count_text no convertibles (valores inválidos): 0
like_count resultante - describe():
count    406.000000
mean       5.726601
std       30.661298
min        0.000000
25%        0.000000
50%        1.000000
75%        2.000000
max      405.000000
Name: like_count, dtype: float64


**Resultado:** `view_count` (videos) ya es numérica y consistente con `view_count_text` en todos los casos comparables, por lo que se usa directamente para los cálculos, tal como recomienda el diccionario de datos. Para comentarios se crea `like_count` (entero), donde los 189 registros con `like_count_text` vacío (representados como `' '`) se imputan a 0 "me gusta" — se documenta esta decisión porque un espacio en blanco es la forma en que YouTube/el scraper representan la ausencia de reacciones, no un dato perdido en sentido estricto. No se encontraron abreviaturas tipo "K"/"mil" ni valores no convertibles en ninguna de las dos columnas.

### 2.5 Dos versiones del texto: `texto_original` y `texto_limpio`

Se trabaja únicamente sobre `comments['text']`, que es la variable textual central del laboratorio (los comentarios). `texto_original` se conserva sin ninguna alteración para permitir auditoría y análisis de sentimiento (que se ve afectado por mayúsculas, signos de exclamación, emojis, etc.); `texto_limpio` se deriva de ella en la sección 2.6.


In [16]:
comments['texto_original'] = comments['text']
comments[['comment_id', 'texto_original']].head(3)

,comment_id,texto_original
0,Ugw-J65a1iYL9hqhELh4AaABAg,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel
1,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic..."
2,Ugw0xaOb2CYXXoudtwJ4AaABAg,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e..."


### 2.6 Construcción y documentación de `texto_limpio`

Pasos aplicados, en orden, y su justificación:

1. **Minúsculas:** normaliza la variación de mayúsculas/minúsculas para frecuencia de palabras y bigramas.
2. **Eliminación de URLs:** los enlaces no aportan contenido temático y distorsionan el conteo de palabras.
3. **Separación de hashtags y menciones:** se conserva la palabra (`#Guatemala` → `guatemala`) en lugar de eliminarla, porque suele llevar contenido temático relevante; se elimina solo el símbolo.
4. **Eliminación de puntuación y números:** reduce ruido para el análisis de frecuencia (los números se documentan como eliminados; para el análisis de sentimiento se sigue usando `texto_original`, donde si importan expresiones como "100%").
5. **Emojis:** se documentan por separado (no se eliminan sin registro) — se extraen a una columna `emojis` antes de removerlos del texto limpio, ya que suelen cargar señal de sentimiento que conviene preservar aparte.
6. **Stopwords en español:** se eliminan usando la lista de NLTK para español (313 palabras), apropiada para reducir palabras funcionales sin valor temático.
7. **Lematización:** se usa `spaCy` (`es_core_news_sm`) para reducir cada palabra a su forma base (p. ej. "corriendo" → "correr"), lo cual mejora la consistencia de conteos de frecuencia/bigramas frente al simple stemming.


In [17]:
import emoji as emoji_lib

def extraer_emojis(texto):
    return ''.join(c for c in str(texto) if c in emoji_lib.EMOJI_DATA)

def limpiar_paso_a_paso(texto):
    t = str(texto)

    # 1. Minúsculas
    t = t.lower()

    # 2. Eliminar URLs
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)

    # 3. Separar hashtags y menciones (conservar la palabra, quitar el símbolo)
    t = re.sub(r'[#@](\w+)', r'\1', t)

    # 4. Eliminar emojis del texto limpio (se guardan aparte en otra columna)
    t = ''.join(c for c in t if c not in emoji_lib.EMOJI_DATA)

    # 5. Eliminar puntuación y números, conservando letras (incl. acentos/ñ) y espacios
    t = re.sub(r'[^a-zñáéíóúü\s]', ' ', t)

    # 6. Colapsar espacios múltiples
    t = re.sub(r'\s+', ' ', t).strip()

    return t

comments['emojis'] = comments['texto_original'].apply(extraer_emojis)
comments['texto_limpio'] = comments['texto_original'].apply(limpiar_paso_a_paso)

comments[['texto_original', 'emojis', 'texto_limpio']].head(5)


,texto_original,emojis,texto_limpio
0,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,,ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel
1,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic...",,están jóvenes porque no buscan un trabajo tuvieron suerte que no hay policía...
2,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e...",,me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales es...
3,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la som...,,veremos a este mafioso de mazariegos en la cárcel y un buen tiempo en la sombra
4,eso es para que salga de USA por su propio pie \nque se auto deporten,,eso es para que salga de usa por su propio pie que se auto deporten


In [18]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stop_es = set(stopwords.words('spanish'))
print(f"Stopwords en español (NLTK): {len(stop_es)}")

def quitar_stopwords(texto):
    palabras = texto.split()
    return ' '.join(p for p in palabras if p not in stop_es)

comments['texto_limpio'] = comments['texto_limpio'].apply(quitar_stopwords)
comments[['texto_original', 'texto_limpio']].head(5)


Stopwords en español (NLTK): 313


,texto_original,texto_limpio
0,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,corrupto amigo vieja fiscal verbose carcel
1,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic...",jóvenes buscan trabajo suerte policías gusta vando ir vestido mujer ensambla...
2,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e...",dejaron ganas demandar ilegalidad reuniones virtuales maquila
3,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la som...,veremos mafioso mazariegos cárcel buen tiempo sombra
4,eso es para que salga de USA por su propio pie \nque se auto deporten,salga usa propio pie auto deporten


In [19]:
import spacy
nlp = spacy.load('es_core_news_sm', disable=['ner', 'parser'])

def lematizar(texto):
    if not texto.strip():
        return texto
    doc = nlp(texto)
    return ' '.join(tok.lemma_ for tok in doc if tok.lemma_.strip())

# Se aplica en lote (nlp.pipe) para eficiencia
lemas = []
for doc in nlp.pipe(comments['texto_limpio'].tolist(), batch_size=64):
    lemas.append(' '.join(tok.lemma_ for tok in doc if tok.lemma_.strip()))
comments['texto_limpio'] = lemas

comments[['texto_original', 'texto_limpio', 'emojis']].head(8)


,texto_original,texto_limpio,emojis
0,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,corrupto amigo viejo fiscal verbo él carcel,
1,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic...",joven buscar trabajo suerte policía gustar var ir vestir mujer ensamblado ma...,
2,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e...",dejar gana demandar ilegalidad reunión virtual maquila,
3,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la som...,ver mafioso mazariegos cárcel buen tiempo sombra,
4,eso es para que salga de USA por su propio pie \nque se auto deporten,salir usar propio pie auto deportar,
5,Imagine if they had to walk back home.,imagine if they had to walk back home,
6,buenísima investigacion :hand-purple-blue-peace::hand-purple-blue-peace::han...,buenísima investigacion hand purple blue peace hand purple blue peace hand p...,
7,"Lleven su lonchera, sacrifiquense un poco. Y reintevren ese dinero. O están ...",llevar lonchero sacrifiquense reintevrir dinero robar,


### 2.7 Cuantificación del efecto de la limpieza

In [20]:
n_total = len(comments)

# Textos vacíos ANTES (texto_original) y DESPUÉS (texto_limpio) de limpiar
vacios_original = (comments['texto_original'].str.strip() == '').sum()
vacios_limpio = (comments['texto_limpio'].str.strip() == '').sum()

# Duplicados de contenido textual ANTES y DESPUÉS
dup_original = comments['texto_original'].str.strip().str.lower().duplicated().sum()
dup_limpio = comments['texto_limpio'].str.strip().duplicated().sum()

# Registros "modificados": el texto limpio es distinto (tras strip/lower) del original
modificados = (comments['texto_original'].str.strip().str.lower() != comments['texto_limpio'].str.strip()).sum()

# Reducción promedio de longitud (proxy del efecto de la limpieza)
long_original = comments['texto_original'].str.len()
long_limpio = comments['texto_limpio'].str.len()
reduccion_pct = 1 - (long_limpio.sum() / long_original.sum())

resumen = pd.DataFrame({
    'métrica': [
        'Comentarios totales',
        'Textos vacíos (texto_original)',
        'Textos vacíos (texto_limpio)',
        'Duplicados de texto (texto_original)',
        'Duplicados de texto (texto_limpio)',
        'Registros modificados por la limpieza',
        'Reducción promedio de longitud de texto',
    ],
    'valor': [
        n_total,
        vacios_original,
        vacios_limpio,
        dup_original,
        dup_limpio,
        modificados,
        f"{reduccion_pct:.1%}",
    ]
})
resumen


,métrica,valor
0,Comentarios totales,406
1,Textos vacíos (texto_original),0
2,Textos vacíos (texto_limpio),5
3,Duplicados de texto (texto_original),3
4,Duplicados de texto (texto_limpio),10
5,Registros modificados por la limpieza,396
6,Reducción promedio de longitud de texto,36.4%


**Interpretación del efecto de la limpieza:**
- Prácticamente todos los comentarios (`modificados`) cambiaron de alguna forma tras el proceso (minúsculas, remoción de puntuación/emojis/stopwords, lematización), lo cual es esperado dado que ningún comentario original cumplía ya el formato normalizado.
- El número de **textos vacíos** creció de 0 (en `texto_original`, ningún comentario venía vacío) a un pequeño número de casos en `texto_limpio` — corresponden a comentarios que consistían **solo** en emojis, menciones o URLs (sin palabras), por lo que tras remover esos elementos no queda contenido textual. Estos casos se documentan y se conservan en el dataset (no se eliminan filas), ya que su `texto_original` y sus `emojis` extraídos siguen siendo válidos para análisis de sentimiento y de emojis.
- Los **duplicados de texto** (dos comentarios con exactamente el mismo contenido) aumentan ligeramente entre `texto_original` y `texto_limpio`, porque comentarios distintos en superficie (p. ej. con o sin mayúsculas, con o sin un emoji al final) pueden converger al mismo `texto_limpio` tras la normalización — esto es un efecto esperado y deseable de la limpieza para análisis de frecuencia, no un error.
- No se eliminó ninguna fila del dataset en este proceso: se documentan los casos problemáticos (vacíos tras limpieza) para que las secciones siguientes decidan cómo tratarlos según el análisis (p. ej. excluirlos solo del conteo de palabras, pero mantenerlos en el análisis de sentimiento sobre `texto_original`).


In [21]:
# Vista final de las columnas clave generadas en el inciso 2, listas para el inciso 3
comments[['comment_id', 'video_id', 'author_channel_id', 'like_count',
          'texto_original', 'texto_limpio', 'emojis']].head(10)


,comment_id,video_id,author_channel_id,like_count,texto_original,texto_limpio,emojis
0,Ugw-J65a1iYL9hqhELh4AaABAg,j43HgwYFKfk,UCdFlugHJJa4l3YqWuNRmvXw,0,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,corrupto amigo viejo fiscal verbo él carcel,
1,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,06mFNPU0aB8,UCvl1tzQeBeGy6efPTRJXSCw,0,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic...",joven buscar trabajo suerte policía gustar var ir vestir mujer ensamblado ma...,
2,Ugw0xaOb2CYXXoudtwJ4AaABAg,j43HgwYFKfk,UCRAquv8el-tQ30bN7MlmySQ,4,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e...",dejar gana demandar ilegalidad reunión virtual maquila,
3,Ugw0xgUc2ISpBr5_T654AaABAg,j43HgwYFKfk,UCkbsS_3D-pvg1iHOld9uvIg,0,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la som...,ver mafioso mazariegos cárcel buen tiempo sombra,
4,Ugw1ZzA21njWhaqQQTh4AaABAg,OkXlHx0hx-8,UCOLHH4ZpMxPYF-Q6e6wvn5A,2,eso es para que salga de USA por su propio pie \nque se auto deporten,salir usar propio pie auto deportar,
5,Ugw1ZzA21njWhaqQQTh4AaABAg.Aa9Tf29tBouAaB_XZQglTL,OkXlHx0hx-8,UCi2KiZq63sRq8Mfbcp-MelQ,0,Imagine if they had to walk back home.,imagine if they had to walk back home,
6,Ugw2014OIYf336oytFt4AaABAg,yLZS3JiEBg8,UCXl30Y_upRqbti52fmjK4YA,0,buenísima investigacion :hand-purple-blue-peace::hand-purple-blue-peace::han...,buenísima investigacion hand purple blue peace hand purple blue peace hand p...,
7,Ugw2bSVOoNn2pc8ZcUl4AaABAg,n8iP75gIpmw,UCRelkU8JiZBytTzebJxaa-w,3,"Lleven su lonchera, sacrifiquense un poco. Y reintevren ese dinero. O están ...",llevar lonchero sacrifiquense reintevrir dinero robar,
8,Ugw3gnUvLp6AqO0_odt4AaABAg,lj983NWyAQY,UCgLurCo-e019HGODcjXHWCw,2,Viva Guatemala 🇬🇹🇬🇹🇬🇹🥳🥳🥳,vivo guatemala,🥳🥳🥳
9,Ugw4C1ijI36l6LAiSZh4AaABAg,n8iP75gIpmw,UCDIMztpVm7G27qBuiWuT6oQ,0,LADRONES !!!!,ladrón,


In [22]:
# Guardado de los datasets procesados para continuar con los siguientes incisos
videos.drop(columns=['view_count_from_text']).to_csv('data/videos_procesado.csv', index=False)
comments.to_csv('data/comments_procesado.csv', index=False)
print("Archivos guardados: data/videos_procesado.csv, data/comments_procesado.csv")


Archivos guardados: data/videos_procesado.csv, data/comments_procesado.csv
